# New Keynesian Model — Julia (DynareJL)

Same 3-equation NK model as the Python notebook.  
Variables: $y_t = [x_t, \pi_t, i_t, r^n_t]$, one shock $\varepsilon_t$.

In [ ]:
# Load DynareJL directly (no Pkg.develop needed)
pkg_src = joinpath(@__DIR__, "..", "julia", "DynareJL", "src")
push!(LOAD_PATH, pkg_src)
include(joinpath(pkg_src, "DynareJL.jl"))
using .DynareJL
using LinearAlgebra, Statistics, Printf

## 1. Parameters

In [ ]:
σ, β, κ, ϕ_π, ϕ_x, ρ = 1.0, 0.99, 0.1, 1.5, 0.5, 0.8

variables = ["x", "pi", "i", "rn"]
shocks    = ["eps_rn"]

## 2. Build ABC matrices

In [ ]:
A = [-1.0  1/σ  0.0  0.0;
      0.0  -β   0.0  0.0;
      0.0   0.0  0.0  0.0;
      0.0   0.0  0.0  0.0]

B = [ 1.0   0.0  1/σ  -1/σ;
     -κ     1.0  0.0   0.0;
     -ϕ_x  -ϕ_π  1.0   0.0;
      0.0   0.0  0.0   1.0]

C = [0.0  0.0  0.0   0.0;
     0.0  0.0  0.0   0.0;
     0.0  0.0  0.0   0.0;
     0.0  0.0  0.0  -ρ  ]

D = [0.0; 0.0; 0.0; -1.0;;]

println("Forward-looking variables (non-zero columns of A):")
println([variables[j] for j in 1:4 if any(A[:, j] .!= 0)])

## 3. Build model and solve

In [ ]:
model  = from_abc(A, B, C, D, variables, shocks)
result = solve(model)

println("=== Blanchard-Kahn Conditions ===")
bk = result.bk
println("  Forward equations (cols of Pi) : $(bk.n_forward)")
println("  Unstable eigenvalues           : $(bk.n_unstable)")
println("  Existence  : $(bk.existence)")
println("  Uniqueness : $(bk.uniqueness)")

## 4. Inspect Schur decomposition

Key transparency feature: eigenvalues, BK matrix, T_hat, Z1.

In [ ]:
sd = result.schur_dec
println("Eigenvalues of G0⁻¹ G1:")
for (i, ev) in enumerate(sd.eigenvalues)
    tag = abs(ev) > 1.01 ? "UNSTABLE" : "stable  "
    println("  λ_$i = $(round(abs(ev), digits=4))  [$tag]")
end
println("\nStable: $(sd.n_stable),  Unstable: $(sd.n_unstable)")

In [ ]:
println("BK matrix Q2 * Pi_bar (should be invertible):")
display(round.(bk.Q2_Pi, digits=4))
println("Singular values: ", round.(svdvals(bk.Q2_Pi), digits=4))

In [ ]:
println("Stable block T̂ (Schur space):")
display(round.(result.T_hat, digits=4))
println("Eigenvalues of T̂: ", sort(abs.(eigvals(result.T_hat)), rev=true) .|> x -> round(x, digits=4))

In [ ]:
# Consistency check: G0 R should equal Psi_eff
Psi_eff = model.Psi + model.Pi * result.eta_matrix
println("max|G0 R - Psi_eff| = ", maximum(abs.(model.G0 * result.R - Psi_eff)),
        "  (should be ~1e-15)")

## 5. IRF

In [ ]:
irf = compute_irf(result, 1; periods=40)
# irf is periods × n_aug_vars

println("IRF (first 10 periods, original 4 variables):")
@printf("%-6s  %-10s %-10s %-10s %-10s\n", "t", "x", "pi", "i", "rn")
for t in 1:10
    @printf("%-6d  %-10.5f %-10.5f %-10.5f %-10.5f\n",
            t-1, irf[t,1], irf[t,2], irf[t,3], irf[t,4])
end

In [ ]:
# Optional plotting with Plots.jl
# import Pkg; Pkg.add("Plots")
# using Plots
# p = plot(layout=(2,2), size=(900,500))
# for (j, var) in enumerate(model.variables)
#     plot!(p[j], irf[:, j], title=var, legend=false, xlabel="Periods")
#     hline!(p[j], [0]; color=:black, linewidth=0.7, linestyle=:dash)
# end
# display(p)

## 6. Stochastic simulation

In [ ]:
sim = simulate(result; periods=500, burn=100, seed=42)
println("Simulation shape: $(size(sim))")

for (j, var) in enumerate(model.variables)
    @printf("%s:  mean=%+.6f,  std=%.6f\n", var, mean(sim[:,j]), std(sim[:,j]))
end

## 7. Forecast Error Variance Decomposition

In [ ]:
decomp = fevd(result; horizon=20, shock_names=shocks)

println("Variance of output gap x at horizons 1, 4, 8, 20:")
for h in [1, 4, 8, 20]
    @printf("  h=%2d:  var(x) = %.8f\n", h, decomp.variance[h, 1])
end

## 8. Z1 — stable mode directions

The policy matrix $T = Z_1 \hat{T} Z_1^\top$.  
Columns of $Z_1$ show which directions in variable space correspond to stable Schur modes.

In [ ]:
Z1 = result.Z1
n_stable = sd.n_stable
println("Z1 (n × n_stable = $(size(Z1,1)) × $n_stable):")
@printf("  %-8s", "var")
for k in 1:n_stable; @printf("  mode_%-4d", k); end; println()
for (i, var) in enumerate(model.aug_variables)
    @printf("  %-8s", var)
    for k in 1:n_stable; @printf("  %+8.4f", Z1[i, k]); end; println()
end

In [ ]:
println("Eta matrix (expectational errors from shocks):")
println("  eta_t = eta_matrix * eps_t")
fwd_vars = [v for v in model.aug_variables if endswith(v, "_fwd")]
for (j, var) in enumerate(fwd_vars)
    @printf("  η(%s):  ", var)
    for k in 1:size(result.eta_matrix, 2)
        @printf("%+10.6f  ", result.eta_matrix[j, k])
    end
    println()
end